#### Pipeline
- 여러 단계( 전처리, 변환, 추정 )를 연속적으로 연결하여 실행하는 도구 
- 전처리 과정과 학습 모델을 하나의 객체(class)로 결합하여 사용
- 교차검증(kfold), 파라미터 탐색(Gridsearch)에서 유용
- 데이터의 누수를 방지 : scaler를 사용하는 경우에는 train data만 자동으로 fit하고 test는 transform
- parameter
    - steps
        - 필수 항목 (기본값이 존재하지 않는다)
        - 파이프 라인의 단계들을 별칭과 객체들을 쌍으로 묶어서 list 형태로 구성
        - ex) [ ( 'scaler', StandardScaler() ), ( 'svm', SVC() ) ]
    - verbose
        - 기본값 : False
        - 각 스탭이 실행이 될때 로그를 출력할것인가? (진행 상황을 표기)
- 속성 
    - name_steps 
        - 파이프라인의 각 단계를 딕셔너리형처럼 접근 하는 방법 
- 메서드
    - fit( X, y, fit_params )
        - 순서대로 각 단계의 fit() 실행
    - fit_transform(X, y, fit_params)
        - 순서대로 각 단계의 transform()을 진행 
    - predict(X)
        - 마지막 단계에서 만들어진 모델에 학습

In [ ]:
import pandas as pd 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
iris = pd.read_csv("../data/iris.csv")
iris.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
le = LabelEncoder()

iris['target'] = le.fit_transform(iris['target'])
iris['target'].value_counts()

In [ ]:
# train. test 데이터셋  분할
x = iris.drop('target', axis=1)
y = iris['target']

X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# 파이프라인 생성 -> 객체 생성
pipe = Pipeline(
    [
        ( 'scaler', StandardScaler() ),     # 1 step
        ( 'model', SVC() )                  # 2 step
    ], verbose= True
)


In [ ]:
pipe

In [ ]:
pipe.fit(X_train, y_train)

In [ ]:
pred = pipe.predict(X_test)

In [ ]:
print(classification_report(y_test, pred))

#### GridSearchCV
- 하이퍼 파라미터(매개변수) 조합을 탐색하여 최적의 조합을 찾는 방법 
- 각 파라미터 조합별로 교차검증(CV)을 수행해 평균 성능을 비교 
- 최적의 모델과 성능을 자동으로 제공 

- parameter
    - estimator
        - 모델의 선택 
        - 학습 모델, Pipeline
        - 모델을 선택할때 고정 파라미터는 모델선택과정에서 지정
    - param_grid
        - 탐색할 파라미터의 조합 (dict 형태로)
        - ex
            - 학습 모델을 선택 : { 'C' : [0.1, 1, 10], 'kernel' : ['linear', 'rbf'] }
            - Pipeline을 선택 : { 'svm__C' : [0.1, 1, 10], 'svm__kernel' : ['linear', 'rbf']  }
    - scoring
        - 기본값 : None
        - 평가 지표 설정 
        - 회귀 모델에 성능 좋은 모델을 찾기 위해서는 neg_mse | mae (오차에 음수형으로 점수를 바꿔주는 작업)
    - cv
        - 기본값 : None
        - 교차 검증의 횟수
        - 5 -> K-Fold(5)
        - train, test 데이터셋을 5회 생성 (회귀에서는 랜덤하게 폴드화 진행)
        - 분류 모델에서는 StratifiedKFold를 이용(자동 전환) -> 계층화 폴드 작업
        - Kfold() 입력값으로 사용 가능 
    - refit
        - 기본값 : True
        - 최적의 파라미터로 전체 데이터를 재 학습할 것인가?
        - True 사용 시 속성에서 best_estimator_을 이용하여 모델을 로드 
        - False 사용 시 속성에서 best_params_를 이용하여 최적의 파라미터를 출력
    - return_train_score
        - 기본값 :  False
        - 교차 검증 시 훈련 성능 점수까지 반영 할것인가?
        - ex
            - train_score // test_score 
                - 두개의 점수가 모두 높다면 -> Best
                - train점수는 높고 test의 점수가 낮은 경우 ->  과적합
                - 두개의 점수가 모두 낮다면 -> 데이터 자체에서 문제이거나 추가적인 데이터 튜닝 필요
    - n_jobs
        - 기본값 : None
        - 병렬 처리할 CPU의 개수를 지정 (-1이면 모든 CPU를 사용)
    - verbose
        - 기본값 : 0
        - 0 : 모든 로그를 표시x 
        - 1 : 로그를 축약해서 표시 (간단하게 보기)
        - 2 : 풀 로그 출력 (상세하게 보기)
    - error_score
        - 기본값 : np.nan(결측치)
        - 모델 학습 시 에러가 발생할때 점수를 어떻게 표현할것인가?(점수에 대한 기본값)
- 속성
    - cv_results
        - 각 파라미터 조합별 성능의 결과 (훈련 / 검증 검수, fit 시간)
    - best_estimator_
        - 최적의 파라미터로 재 학습이 된 모델 객체
    - bset_parmas_
        - 최적의 성능을 낸 파라미터들의 조합
    - best_score_
        - 최적의 파라미터로 조합이 된 모델의 교차 검증 평균 성능
    - refit_time_
        - 재 학습에 걸린 시간
- 메서드
    - fit() 
        - 하이퍼 파라미터 조합별 교차 검증 시작
    - predict()
        - 최적의 모델을 이용하여 예측
    - predict_proba()
        - 분류인 경우 확률 예측
    - score
        - 최적의 모델의 점수를 출력

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
# 검증
# SVC 모델을 생성 
svc = SVC()
# svc에서 사용할 매개변수의 목록들을 작성 
grid_dict = {
    'C' : [0.1, 1, 10], 
    'kernel' : ['linear', 'rbf'], 
    'gamma' : ['scale', 'auto']
}

grid = GridSearchCV(
    estimator= svc, 
    param_grid= grid_dict, 
    cv = 5, 
    scoring = 'accuracy',
    verbose=1, 
    refit = True

)

In [ ]:
grid.fit(X_train, y_train)

In [ ]:
print("최적의 파라미터 조합 : ", grid.best_params_)
print("최적의 스코어 : ", grid.best_score_)
print("최적의 모델 : ", grid.best_estimator_)

#### KFold
- 데이터셋을 K개의 동일 크기 부분(Fold)으로 나눠서 K번 반복하여 하나의 폴드를 검증용으로 사용을 하고 나머지 K-1개의 폴드를 학습용으로 사용
- 모든 폴드를 한번씩은 검증용으로 포함을 시켜서 성능을 안정적으로 평가 할 수 있는 방법
- 데이터의 개수가 적은 경우 반복 실행 작업을 이용하여 성능을 안정적으로 평가

- parameter
    - n_splits
        - 기본값 : 5
        - 폴드의 개수를 지정 
        - 최소 값은 2
    - shuffle
        - 기본값 : False
        - 데이터를 분할하기 전에 데이터들을 섞을지 지정
        - True로 변경하면 폴드 안에 데이터가 랜덤하게 구성 
    - random_state
        - 기본값 : None
        - shuffle이 True인 경우에만 사용
        - 랜덤 시드를 고정
- 속성
    - n_splits
        - 분할된 폴드의 개수 
- 메서드 
    - split(X, y = None)
        - 학습용 검증용 인덱스를 생성 
        - 반복문을 이용하여 (train_index, test_index)로 변환하여 사용

- 장점
    - 데이터를 폴드화해서 학습/검증을 사용하기 때문에 데이터의 낭비가 없다. 
    - train_test_split에 비해서 성능 평가 안정적
- 단점
    - K번의 학습 -> K번의 예측 -> K번의 평가 : 계산 횟수가 늘어난다. 시간이 증가
    - 데이터의 규모가 커지면 커질수록 시간은 크게 증가

- 특수한 경우 변형 KFold 종류
    - StratifiedKFold : 분류 문제에거 클래스의 비율을 유지하며 분할 
    - GroupKFold : 그룹 단위로 데이터를 나눠 특정 그룹으로 학습하고 다른 그룹으로 검증을 하는 방법
    - RepeatedKFold : KFold를 여러번 반복해 평가 안정성 강화



In [ ]:
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold

In [ ]:
X = np.array(
    ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J']
)

# KFold 
kfold = KFold( n_splits=2, shuffle=True, random_state=42)

# list(kfold.split(X))

for train_idx, test_idx in kfold.split(X):
    # print(train_idx)
    print('-' * 60)
    print("학습 데이터의 목록 : ", X[train_idx])
    print('검증 데이터의 목록 : ', X[test_idx])

In [ ]:
y = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
# list(kfold.split(X, y))
for train_idx, test_idx in kfold.split(X):
    # print(train_idx)
    print('-' * 60)
    print("학습 데이터의 목록 : ", X[train_idx])
    print("학습 정답의 목록 : ", y[train_idx])
    print('검증 데이터의 목록 : ', X[test_idx])
    print('검증 정답의 목록 : ', y[test_idx])

In [ ]:
from sklearn.svm import SVR

In [ ]:
x = iris.drop('target', axis=1)
y = iris['target']

In [ ]:
# Pipeline + GridSearchCV + StratifiedKFold 
cv_fold = StratifiedKFold(shuffle=True, random_state=42)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# Pipeline 생성 
pipe = Pipeline(
    [
        ('scaler', StandardScaler()), 
        ('clf', SVC(random_state=42, probability=True))
    ]
)

In [ ]:
# clf에서 사용할 파라미터 조합 생성 
# GridSearchCV에서 사용할 파라미터 조합 ( 별칭__매개변수명 )
params = {
    'clf__C' : [0.1, 1, 10], 
    'clf__gamma' : ['scale', 'auto'], 
    'clf__kernel' : ['linear', 'rbf']
}

In [ ]:
grid_clf = GridSearchCV(
    estimator= pipe,        # 파이프라인으로 생성한 모델을 지정
    param_grid= params,     # dict 형태로 파라미터의 조합 
    scoring= 'accuracy',    # 베스트 모델 선정 기준
    cv = cv_fold,           # 생성해둔 KFold 지정
    refit = True,           # 재학습 여부
    n_jobs = -1,            # 모든 CPU를 사용
    return_train_score= True,   # 학습 데이터의 성능을 출력
    verbose= 2              # 로그 표시를 상세하게 
)

In [ ]:
grid_clf.fit(X_train, y_train)

In [ ]:
grid_clf.score(X_test, y_test)

In [ ]:
grid_df = pd.DataFrame(grid_clf.cv_results_)
grid_df.sort_values('mean_test_score', ascending=False)

In [ ]:
print(grid_clf.best_params_)

In [ ]:
svc_final = SVC(C = 0.1, gamma = 'scale', kernel = 'linear', random_state=42)

In [ ]:
svc_final.fit(X_train, y_train)

In [ ]:
pred = svc_final.predict(X_test)

In [ ]:
print(classification_report(y_test, pred))

- 연습 
1. boston 데이터셋을 로드 
2. 독립변수, 종속변수로 데이터를 분할 
3. train, test로 8:2 비율로 변환 
4. KFold을 이용하여 10개의 폴드를 생성 (shuffle True)
5. 파이프라인 생성 ( StandardScaler(),  SVR()  )
6. 파라미터 조합을 생성 ( svr모델에서 C ( 1, 10, 100 ), kernel ( 'linear', 'rbf' ) , epsilon ( 0.1, 0.2, 0.5 ) )
7. GridSearchCV 베스트 모델 선정의 기준은 neg_mean_squared_error
8. 최고의 조합을 찾은 뒤 R2-score를 이용하여 성능을 확인

In [ ]:
boston = pd.read_csv("../csv/boston.csv")

In [ ]:
x = boston.drop('Price', axis=1)
y = boston['Price']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [ ]:
kfold = KFold(n_splits=10, shuffle = True, random_state=42)

In [ ]:
pipe_reg = Pipeline(
    [
        ('scaler', StandardScaler()), 
        ('reg', SVR())
    ]
)

In [ ]:
params_reg = {
    "reg__C" : [1, 10, 100], 
    'reg__kernel' : ['linear', 'rbf'], 
    "reg__epsilon" : [0.1, 0.2, 0.5]
}

In [ ]:
grid_reg = GridSearchCV(
    estimator= pipe_reg, 
    param_grid= params_reg, 
    scoring= 'neg_mean_squared_error', 
    cv = kfold, 
    verbose=1, 
    refit = True
)

In [ ]:
grid_reg.fit(X_train, y_train)

In [77]:
# 최적의 모델이 저장되어있는 속성 : best_estimator_ 
grid_reg.best_estimator_.score(X_test, y_test)

0.8354920503211448

In [78]:
grid_reg.score(X_test, y_test)

-12.063990309898486

In [79]:
from sklearn.metrics import r2_score

In [80]:
pipe_pred = grid_reg.best_estimator_.predict(X_test)
grid_pred = grid_reg.predict(X_test)

In [81]:
print(r2_score(y_test, pipe_pred))
print(r2_score(y_test, grid_pred))

0.8354920503211448
0.8354920503211448
